# 第 5 节：Bellman 方程与动态规划

---

## 📍 本节位置

```
MDP (04) → **Bellman/DP (05)** → MC (06) → TD (07) → ...
                ↑
            你在这里
```

本节是 RL 理论核心中的核心。如果你只能彻底理解一节，就选这一节。

---

## 🎯 学习目标

1. 理解状态价值函数 $V^\pi(s)$ 和动作价值函数 $Q^\pi(s,a)$ 的定义
2. 推导 Bellman 期望方程（矩阵形式和逐状态形式）
3. 推导 Bellman 最优方程
4. 理解 Bellman operator 及其 contraction 性质的直觉
5. 实现 Policy Evaluation（策略评估）
6. 实现 Policy Iteration（策略迭代）
7. 实现 Value Iteration（价值迭代）
8. 在小 GridWorld 上数值验证所有算法


## 1. 状态价值函数 $V^\pi(s)$

### 定义

在策略 $\pi$ 下，状态 $s$ 的价值定义为从 $s$ 出发、遵循 $\pi$ 的**期望回报**：

$$V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s] = \mathbb{E}_\pi\left[\sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \;\middle|\; S_t = s\right]$$

- $V^\pi(s)$ 是**状态**的函数（不涉及具体动作）
- 期望 $\mathbb{E}_\pi$ 表示：动作按 $\pi$ 选择，状态按 $P$ 转移
- $V^\pi$ 衡量"在这个状态下，预期能获得多少累积奖励"

### 手算示例

考虑一个简单的 2 状态 MDP。让我们用代码计算 $V^\pi$。


In [ ]:
import numpy as np
import sys
sys.path.insert(0, '/workspace/data/vggt-omega/rl')
from rl_course.utils.seeding import set_seed
set_seed(42)

# 一个简单 3 状态链式 MDP
# s0 → s1 → s2 (终止), 每一步 reward=+1 或 -1
# 动作: 0=左, 1=右

n_states, n_actions = 3, 2
gamma = 0.9

# 转移矩阵: P[s, a, s']
P = np.zeros((n_states, n_actions, n_states))
P[0, 1, 1] = 1.0  # s0 向右 → s1
P[1, 1, 2] = 1.0  # s1 向右 → s2
P[1, 0, 0] = 1.0  # s1 向左 → s0
P[2, :, 2] = 1.0  # s2 吸收态

# 奖励矩阵: R[s, a]
R = np.zeros((n_states, n_actions))
R[0, 1] = 1.0  # s0→s1 获得 +1
R[1, 1] = 2.0  # s1→s2 获得 +2
R[1, 0] = -1.0  # s1→s0 获得 -1

# 策略: 随机策略 π(a|s) = 0.5
pi = np.ones((n_states, n_actions)) / n_actions

print("3 状态链式 MDP:")
print(f"P shape: {P.shape}, R shape: {R.shape}")
print(f"策略 π = {pi}")


## 2. 动作价值函数 $Q^\pi(s,a)$

### 定义

在策略 $\pi$ 下，在状态 $s$ 执行动作 $a$ 后，后续遵循 $\pi$ 的期望回报：

$$Q^\pi(s, a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a] = \mathbb{E}_\pi\left[\sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \;\middle|\; S_t = s, A_t = a\right]$$

### V 和 Q 的关系

$$V^\pi(s) = \sum_{a \in \mathcal{A}} \pi(a|s) Q^\pi(s, a)$$

$$Q^\pi(s, a) = R(s, a) + \gamma \sum_{s' \in \mathcal{S}} P(s'|s, a) V^\pi(s')$$

**直觉**：$V$ 是对动作求期望后的价值，$Q$ 是给定具体动作后的价值。


## 3. Bellman 期望方程

### 推导

从 $V^\pi$ 的定义出发，利用 $G_t = R_{t+1} + \gamma G_{t+1}$：

$$
\begin{aligned}
V^\pi(s) &= \mathbb{E}_\pi[G_t \mid S_t = s] \\
&= \mathbb{E}_\pi[R_{t+1} + \gamma G_{t+1} \mid S_t = s] \\
&= \sum_{a} \pi(a|s) \sum_{s'} P(s'|s,a) \left[ R(s,a) + \gamma \, \mathbb{E}_\pi[G_{t+1} \mid S_{t+1}=s'] \right] \\
&= \sum_{a} \pi(a|s) \sum_{s'} P(s'|s,a) \left[ R(s,a) + \gamma \, V^\pi(s') \right]
\end{aligned}
$$

### 两种形式

**逐状态形式**（用于编程实现）：

$$V^\pi(s) = \sum_{a} \pi(a|s) \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^\pi(s') \right]$$

**矩阵形式**（用于理论分析）：

$$\mathbf{V}^\pi = \mathbf{R}^\pi + \gamma \mathbf{P}^\pi \mathbf{V}^\pi$$

其中 $\mathbf{R}^\pi_s = \sum_a \pi(a|s)R(s,a)$，$\mathbf{P}^\pi_{s,s'} = \sum_a \pi(a|s)P(s'|s,a)$。

### 解析解

从矩阵形式可以解出：

$$\mathbf{V}^\pi = (\mathbf{I} - \gamma \mathbf{P}^\pi)^{-1} \mathbf{R}^\pi$$

这说明 Bellman 方程是一个**线性方程组**。


### 3.1 代码验证：解析解 vs 迭代解

In [ ]:
def bellman_expectation_solve(P, R, pi, gamma):
    '''矩阵形式解析求解 V^π = (I - γP^π)^(-1) R^π'''
    n_states = P.shape[0]

    # 构建策略下的转移矩阵和奖励向量
    P_pi = np.zeros((n_states, n_states))
    R_pi = np.zeros(n_states)

    for s in range(n_states):
        for a in range(n_actions):
            P_pi[s] += pi[s, a] * P[s, a]  # shape (n_states,)
            R_pi[s] += pi[s, a] * R[s, a]

    # 解析解
    I = np.eye(n_states)
    V = np.linalg.solve(I - gamma * P_pi, R_pi)
    return V, P_pi, R_pi

V_solved, P_pi, R_pi = bellman_expectation_solve(P, R, pi, gamma)
print("解析解 V^π:")
for s in range(n_states):
    print(f"  V({s}) = {V_solved[s]:.4f}")

# 迭代验证（下一节的 Policy Evaluation）
V_iter = np.zeros(n_states)
for i in range(1000):
    V_new = np.zeros(n_states)
    for s in range(n_states):
        for a in range(n_actions):
            prob = pi[s, a]
            for ns in range(n_states):
                V_new[s] += prob * P[s, a, ns] * (R[s, a] + gamma * V_iter[ns])
    if np.max(np.abs(V_new - V_iter)) < 1e-10:
        print(f"迭代法在第 {i+1} 步收敛")
        break
    V_iter = V_new

print(f"\n解析解: {V_solved}")
print(f"迭代解: {V_iter}")
print(f"最大误差: {np.max(np.abs(V_solved - V_iter)):.2e}")


## 4. Bellman 最优方程

### 最优价值函数

**最优状态价值函数**：在所有策略中取最大

$$V^*(s) = \max_\pi V^\pi(s)$$

**最优动作价值函数**：

$$Q^*(s, a) = \max_\pi Q^\pi(s, a)$$

### Bellman 最优方程

$$V^*(s) = \max_a \left[ R(s, a) + \gamma \sum_{s'} P(s'|s, a) V^*(s') \right]$$

$$Q^*(s, a) = R(s, a) + \gamma \sum_{s'} P(s'|s, a) \max_{a'} Q^*(s', a')$$

**关键点**：最优方程中的 $\max$ 意味着我们不是评估某个策略，而是直接寻找最优策略。

### Bellman Operator 与 Contraction

定义 **Bellman 最优算子** $\mathcal{T}$：

$$(\mathcal{T} V)(s) = \max_a \left[ R(s, a) + \gamma \sum_{s'} P(s'|s, a) V(s') \right]$$

$\mathcal{T}$ 是一个 **收缩映射（Contraction）**：

$$\|\mathcal{T} V_1 - \mathcal{T} V_2\|_\infty \leq \gamma \|V_1 - V_2\|_\infty$$

**直觉**：每次应用 $\mathcal{T}$，误差至少缩小 $\gamma$ 倍。重复应用保证收敛到唯一不动点 $V^*$。

这是 **Value Iteration** 收敛性的理论基础。


## 5. Policy Evaluation（策略评估）

### 算法

给定策略 $\pi$，通过迭代计算 $V^\pi$：

1. 初始化 $V(s) = 0$（或任意值）对所有 $s$
2. 重复：
   - $\Delta \leftarrow 0$
   - 对每个状态 $s$：
     - $v \leftarrow V(s)$
     - $V(s) \leftarrow \sum_a \pi(a|s) \sum_{s'} P(s'|s,a)[R(s,a) + \gamma V(s')]$
     - $\Delta \leftarrow \max(\Delta, |v - V(s)|)$
3. 直到 $\Delta < \theta$（收敛阈值）

这个算法也被称为 **Iterative Policy Evaluation**。


In [ ]:
def policy_evaluation(P, R, pi, gamma, theta=1e-6, max_iter=10000):
    '''迭代策略评估

    Args:
        P: 转移矩阵 (nS, nA, nS)
        R: 奖励矩阵 (nS, nA)
        pi: 策略 (nS, nA)
        gamma: 折扣因子
        theta: 收敛阈值
        max_iter: 最大迭代次数

    Returns:
        V: 状态价值 (nS,)
        history: 每次迭代的 V 值（用于可视化）
    '''
    nS, nA = P.shape[0], P.shape[1]
    V = np.zeros(nS)
    history = [V.copy()]

    for iteration in range(max_iter):
        delta = 0.0
        V_new = np.zeros(nS)

        for s in range(nS):
            v = V[s]
            # Bellman 期望方程的一次应用
            new_v = 0.0
            for a in range(nA):
                for ns in range(nS):
                    new_v += pi[s, a] * P[s, a, ns] * (R[s, a] + gamma * V[ns])
            V_new[s] = new_v
            delta = max(delta, abs(v - V_new[s]))

        V = V_new
        history.append(V.copy())

        if delta < theta:
            print(f"Policy Evaluation 收敛于第 {iteration+1} 次迭代 (Δ={delta:.2e})")
            break

    return V, history

# 在 3 状态链式 MDP 上测试
V_pe, hist = policy_evaluation(P, R, pi, gamma)
print(f"\nPolicy Evaluation 结果: {V_pe}")
print(f"与解析解的误差: {np.max(np.abs(V_pe - V_solved)):.2e}")


## 6. Policy Iteration（策略迭代）

### 核心思想

交替执行两个步骤直至收敛：

1. **Policy Evaluation**：计算当前策略 $\pi_k$ 的 $V^{\pi_k}$
2. **Policy Improvement**：对每个状态 $s$，选择贪心动作

$$\pi_{k+1}(s) = \arg\max_a \sum_{s'} P(s'|s,a)[R(s,a) + \gamma V^{\pi_k}(s')]$$

### 为什么保证改进？

**Policy Improvement Theorem**：如果对所有 $s$ 有 $Q^{\pi}(s, \pi'(s)) \geq V^{\pi}(s)$，则 $V^{\pi'} \geq V^{\pi}$。

贪心选择满足这个条件，所以每次策略更新都不会变差。

### 算法流程

```
1. 初始化 π₀ (随机策略)
2. 重复:
   a. V = PolicyEvaluation(π_k)     ← 评估
   b. 对每个状态 s:
      π_{k+1}(s) = argmax_a Σ_s' P(s'|s,a)[R(s,a) + γ V(s')]  ← 改进
   c. 如果 π_{k+1} == π_k: 停止
```


In [ ]:
def policy_improvement(P, R, V, gamma):
    '''策略改进：对每个状态选择贪心动作

    Returns:
        new_pi: 新策略 (nS, nA) — 确定性策略
        improved: 策略是否真的改进了
    '''
    nS, nA = P.shape[0], P.shape[1]
    pi_new = np.zeros((nS, nA))

    for s in range(nS):
        q_values = np.zeros(nA)
        for a in range(nA):
            for ns in range(nS):
                q_values[a] += P[s, a, ns] * (R[s, a] + gamma * V[ns])
        best_action = np.argmax(q_values)
        pi_new[s, best_action] = 1.0

    return pi_new


def policy_iteration(P, R, gamma, theta=1e-6, max_iter=100):
    '''完整的 Policy Iteration 算法

    Returns:
        V: 最优状态价值
        pi: 最优策略
        history: 每轮迭代的策略 (用于可视化)
    '''
    nS, nA = P.shape[0], P.shape[1]
    pi = np.ones((nS, nA)) / nA  # 初始随机策略
    pi_history = [pi.copy()]

    for iteration in range(max_iter):
        # 步骤 1: Policy Evaluation
        V, _ = policy_evaluation(P, R, pi, gamma, theta)

        # 步骤 2: Policy Improvement
        pi_new = policy_improvement(P, R, V, gamma)

        pi_history.append(pi_new.copy())

        # 检查策略是否稳定
        if np.array_equal(pi_new, pi):
            print(f"Policy Iteration 收敛于第 {iteration+1} 轮")
            break

        pi = pi_new

    return V, pi, pi_history

# 在 3 状态链式 MDP 上测试
V_opt, pi_opt, pi_history = policy_iteration(P, R, gamma)
print(f"\n最优价值 V*: {V_opt}")
print(f"最优策略:")
for s in range(n_states):
    best_a = np.argmax(pi_opt[s])
    print(f"  π({s}) = {best_a}")


## 7. Value Iteration（价值迭代）

### 核心思想

将 Policy Evaluation 截断为**一次更新**，然后直接做贪心改进。

也就是：将 Bellman 最优算子反复应用到 $V$ 上直到收敛。

### 算法

```
1. 初始化 V(s) = 0
2. 重复:
   Δ = 0
   对每个状态 s:
     v = V(s)
     V(s) = max_a Σ_{s'} P(s'|s,a)[R(s,a) + γ V(s')]
     Δ = max(Δ, |v - V(s)|)
3. 直到 Δ < θ
4. 从 V* 中提取最优策略 π*
```

### 与 Policy Iteration 的对比

| 方面 | Policy Iteration | Value Iteration |
|------|-----------------|-----------------|
| 评估步 | 多次迭代至收敛 | 仅一次更新 |
| 每轮计算量 | 大 | 小 |
| 收敛轮数 | 少 | 多 |
| 总时间 | 取决于 MDP | 取决于 MDP（通常更快） |


In [ ]:
def value_iteration(P, R, gamma, theta=1e-6, max_iter=10000):
    '''Value Iteration 算法

    Returns:
        V: 最优状态价值
        history: V 的演化历史
        deltas: 每轮的 Δ（收敛速度可视化）
    '''
    nS, nA = P.shape[0], P.shape[1]
    V = np.zeros(nS)
    history = [V.copy()]
    deltas = []

    for iteration in range(max_iter):
        delta = 0.0
        V_new = np.zeros(nS)

        for s in range(nS):
            # 计算所有动作的 Q 值，取最大
            q_values = np.zeros(nA)
            for a in range(nA):
                for ns in range(nS):
                    q_values[a] += P[s, a, ns] * (R[s, a] + gamma * V[ns])
            V_new[s] = np.max(q_values)
            delta = max(delta, abs(V[s] - V_new[s]))

        V = V_new
        history.append(V.copy())
        deltas.append(delta)

        if delta < theta:
            print(f"Value Iteration 收敛于第 {iteration+1} 次迭代 (Δ={delta:.2e})")
            break

    # 提取最优策略
    pi_opt = np.zeros((nS, nA))
    for s in range(nS):
        q_values = np.zeros(nA)
        for a in range(nA):
            for ns in range(nS):
                q_values[a] += P[s, a, ns] * (R[s, a] + gamma * V[ns])
        pi_opt[s, np.argmax(q_values)] = 1.0

    return V, pi_opt, history, deltas

# 测试
V_vi, pi_vi, hist_vi, deltas = value_iteration(P, R, gamma)
print(f"\n最优价值 V*: {V_vi}")
print(f"最优策略:")
for s in range(n_states):
    best_a = np.argmax(pi_vi[s])
    print(f"  π({s}) = {best_a}")

# 对比两种方法
print(f"\nPolicy Iteration  vs  Value Iteration:")
print(f"V* (PI):  {V_opt}")
print(f"V* (VI):  {V_vi}")
print(f"误差:    {np.max(np.abs(V_opt - V_vi)):.2e}")


## 8. GridWorld 实验

在更大的 GridWorld 上运行 Policy Iteration 和 Value Iteration。


In [ ]:
from rl_course.envs.grid_world import GridWorld
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 创建 5x5 GridWorld
gw = GridWorld(width=5, height=5, step_reward=-1.0, goal_reward=10.0, seed=42)
P_gw = gw.get_transition_matrix()
R_gw = gw.get_reward_matrix()
gamma = 0.99

print(f"GridWorld: {gw.n_states} 状态, {gw.n_actions} 动作")

# Value Iteration
V_vi, pi_vi, hist_vi, deltas = value_iteration(P_gw, R_gw, gamma)

# 可视化收敛过程
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
iterations_to_show = [0, 1, 2, 5, 10, len(hist_vi)-1]
for ax, it in zip(axes.flatten(), iterations_to_show):
    v_grid = hist_vi[it].reshape(5, 5)
    im = ax.imshow(v_grid, cmap='YlOrRd')
    ax.set_title(f'Iteration {it}')
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f'{v_grid[i,j]:.1f}', ha='center', va='center', fontsize=8)
    plt.colorbar(im, ax=ax)
plt.suptitle('Value Iteration 收敛过程', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/figures/05_value_iteration_convergence.png', dpi=100)
plt.close()
print("✅ Value Iteration 收敛过程图已保存")

# 最优策略可视化
from rl_course.visualization.plotting import plot_policy_grid, plot_value_heatmap

# 提取最优动作
best_actions = np.argmax(pi_vi, axis=1)
plot_policy_grid(best_actions, (5, 5), n_actions=4,
                 title='GridWorld 最优策略 (Value Iteration)',
                 filepath='outputs/figures/05_gridworld_policy.png')
print("✅ 最优策略图已保存")

plot_value_heatmap(V_vi, (5, 5),
                   title='GridWorld 最优状态价值 V*',
                   filepath='outputs/figures/05_gridworld_value.png')
print("✅ 最优价值热力图已保存")


## 9. Policy Iteration vs Value Iteration 对比实验

In [ ]:
# 更复杂的 GridWorld：有障碍物
gw2 = GridWorld(
    width=6, height=6,
    start_pos=(0, 0), goal_pos=(5, 5),
    blocked_positions=[(2, 2), (2, 3), (3, 2)],
    step_reward=-0.1, goal_reward=10.0,
    seed=42
)
P2 = gw2.get_transition_matrix()
R2 = gw2.get_reward_matrix()

import time

# Policy Iteration
t0 = time.time()
V_pi, pi_pi, _ = policy_iteration(P2, R2, gamma)
t_pi = time.time() - t0
print(f"Policy Iteration: {t_pi:.3f}s")

# Value Iteration
t0 = time.time()
V_vi, pi_vi, _, deltas_vi = value_iteration(P2, R2, gamma)
t_vi = time.time() - t0
print(f"Value Iteration: {t_vi:.3f}s")

# 收敛速度对比
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(deltas_vi, linewidth=1.5)
ax.set_yscale('log')
ax.set_xlabel('Iteration')
ax.set_ylabel('Δ (log scale)')
ax.set_title('Value Iteration 收敛速度')
ax.grid(True, alpha=0.3)
plt.savefig('outputs/figures/05_vi_convergence_rate.png', dpi=100)
plt.close()

# 最优策略验证
best_pi = np.argmax(pi_pi, axis=1)
best_vi = np.argmax(pi_vi, axis=1)
assert np.array_equal(best_pi, best_vi), "两种方法的最优策略应该一致"
print("\n✅ Policy Iteration 和 Value Iteration 的最优策略一致")


## 10. 本节总结

### 核心方程

| 方程 | 公式 | 用途 |
|------|------|------|
| Bellman 期望方程 | $V^\pi(s) = \sum_a \pi(a\vert s) \sum_{s'} P(s'\vert s,a)[R + \gamma V^\pi(s')]$ | 评估给定策略 |
| Bellman 最优方程 | $V^*(s) = \max_a \sum_{s'} P(s'\vert s,a)[R + \gamma V^*(s')]$ | 寻找最优策略 |
| Contraction | $\|\mathcal{T}V_1 - \mathcal{T}V_2\|_\infty \leq \gamma \|V_1 - V_2\|_\infty$ | 保证迭代收敛 |

### 算法对比

| 算法 | 思路 | 每轮开销 | 收敛轮数 |
|------|------|----------|----------|
| Policy Evaluation | 给定 π，迭代计算 V^π | O(nS² × nA) | 多 |
| Policy Iteration | 交替评估+改进 | O(nS² × nA × n_iter) | 少 |
| Value Iteration | 直接迭代 Bellman 最优方程 | O(nS² × nA) | 中 |

### 关键理解

1. **Bellman 方程是自洽的**：$V$ 的值取决于 $V$ 自己，这形成了不动点方程
2. **γ < 1 保证 contraction**：这是所有 DP 算法收敛的根本原因
3. **DP 需要完整模型**：必须知道 $P$ 和 $R$，这是 DP 的局限
4. **后续的 MC、TD 等方法在不知道 $P$ 和 $R$ 的情况下近似求解 Bellman 方程**


## 11. 面试问题

<details>
<summary><b>Q1: 解释 Bellman 方程及其重要性</b></summary>

Bellman 方程将价值函数分解为**即时奖励** + **折扣后的未来价值**：
$$V(s) = \max_a [R(s,a) + \gamma \sum_{s'} P(s'|s,a)V(s')]$$

它是 RL 的核心，因为它将"优化整个未来"这个问题分解为"优化当前决策 + 继续优化后续"的子问题（最优性原理）。
</details>

<details>
<summary><b>Q2: Policy Iteration 和 Value Iteration 的区别？</b></summary>

- **PI**：显式维护策略。每轮完整评估当前策略，然后改进。评估多轮，改进少轮。
- **VI**：不显式维护策略。每次更新直接取 max。本质是将 PI 的评估步骤截断为一次。

在实践中，VI 通常更快且实现更简单。
</details>

<details>
<summary><b>Q3: 什么是 Bellman operator 的 contraction？</b></summary>

Contraction 指算子 $\mathcal{T}$ 使两个值函数之间的距离缩小至少 $\gamma$ 倍：
$$\|\mathcal{T}V_1 - \mathcal{T}V_2\| \leq \gamma\|V_1 - V_2\|$$

这是 Banach 不动点定理的应用——重复应用必收敛到唯一不动点 $V^*$。
</details>


## 12. 练习

### 概念题
1. 手算推导一个 2×2 GridWorld 的 Bellman 期望方程（写出具体的线性方程组）
2. 证明：如果 $\gamma=1$ 且 MDP 有环，$V^\pi$ 可能发散

### 编程题
3. 修改 GridWorld 的奖励函数（如每步 reward 改为 -0.01），观察最优策略是否改变
4. 实现 Gauss-Seidel Value Iteration（异步更新，更快收敛），与标准 VI 对比
5. 在随机 GridWorld (slip_prob > 0) 上计算最优策略，观察其与确定性版本的区别


---
*下一节：[06_monte_carlo.ipynb](06_monte_carlo.ipynb) — Monte Carlo 方法（从已知模型到从经验学习）*
